# Collaborator-Matched Poisson Ridge Pipeline

Step-by-step walkthrough of the updated semantic encoding GLM pipeline, demonstrating:

1. **Per-fold PCA** — standardise → PCA → re-standardise, all fit on train only  
2. **Nested CV** — 5-fold outer, 5-fold inner; per-neuron alpha selection  
3. **McFadden's pseudo-R²** — `1 - LL_model / LL_null` (complete Poisson LL with log-factorial)  
4. **X-permutation significance** — shuffle embedding matrix globally, refit per permutation  
5. **Result saving** — per-patient pkl + aggregated pkl with `ll_real`, `ll_null` columns

This notebook runs on a single patient + layer to illustrate the pipeline.
The full pipeline is in `scripts/semantic_glm.py`.

In [ ]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import torch
from scipy.special import gammaln

warnings.filterwarnings('ignore')
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')

PROJECT = '/scratch/aniluchavez/hippocampal-speaker-semantics'
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f'  {p.name}  {p.total_memory//1024**2} MB')

## 1. Configuration

In [ ]:
# ── Choose patient / model / layer ─────────────────────────────────────────
PATIENT_ID  = 'PTYEU_task147'
PATIENT     = 'ptYEU_task147'
REGION      = 'hippocampus'
CONDITION   = 'self'           # 'self' or 'other'
MODEL_TAG   = 'gpt2-xl'
CONTEXT_TAG = '_ctx200'
LAYER       = 24               # gpt2-xl best layer from sweep
WINDOW_TYPE = 'varwin'         # 'fixed' or 'varwin'

# ── Pipeline hyperparameters (matching collaborators) ──────────────────────
ALPHAS      = np.logspace(-3, 3, 30)
N_COMPONENTS = 100
N_OUTER     = 5
N_INNER     = 5
N_PERM      = 500
SEED        = 42
LBFGS_ITER  = 200
LBFGS_ITER_PERM = 100
ETA_CLIP    = 20.0
MIN_SPIKES  = 5
FDR_ALPHA   = 0.05
USE_OFFSET  = WINDOW_TYPE == 'varwin'

# ── Paths ──────────────────────────────────────────────────────────────────
EMBED_DIR  = '/scratch/aniluchavez/ConvoDATAS/EmbedCache'
SPIKE_ROOT = '/scratch/aniluchavez/ConvoDATAS/SpikeWindows'
WIN_TAG    = 'varwin_m150p500_p200p500' if USE_OFFSET else 'tshift-150_tlen500_oshift+200_olen500'
SPIKE_DIR  = os.path.join(SPIKE_ROOT, f'output_{PATIENT}_english_only_{WIN_TAG}')

npy_path = os.path.join(EMBED_DIR,
    f'{PATIENT_ID}_{MODEL_TAG}{CONTEXT_TAG}_word_emb_layers.npy')

print('Embeddings:', npy_path)
print('Spike dir: ', SPIKE_DIR)
print('Exists:', os.path.exists(npy_path), '/', os.path.isdir(SPIKE_DIR))

## 2. Helper Functions

In [ ]:
# ── PCA helpers ────────────────────────────────────────────────────────────

@torch.no_grad()
def gpu_pca_fit(X_np, n_components):
    """Randomised SVD PCA on GPU. Returns (X_pca, train_mean, Vt)."""
    X  = torch.tensor(X_np, dtype=torch.float32, device=DEVICE)
    Xm = X.mean(0)
    Xc = X - Xm
    k  = min(n_components + 10, min(Xc.shape))
    Y  = Xc @ torch.randn(Xc.shape[1], k, device=DEVICE)
    for _ in range(2):
        Y = Xc @ (Xc.T @ Y)
    Q, _ = torch.linalg.qr(Y)
    _, _, Vt = torch.linalg.svd(Q.T @ Xc, full_matrices=False)
    Vt = Vt[:n_components]
    X_pca = (Xc @ Vt.T).cpu().numpy().astype(np.float32)
    return X_pca, Xm.cpu().numpy(), Vt.cpu().numpy()


@torch.no_grad()
def gpu_pca_transform(X_np, train_mean, Vt_np):
    X  = torch.tensor(X_np - train_mean, dtype=torch.float32, device=DEVICE)
    Vt = torch.tensor(Vt_np, dtype=torch.float32, device=DEVICE)
    return (X @ Vt.T).cpu().numpy().astype(np.float32)


def prep_fold_features(X_tr_raw, X_te_raw, n_components):
    """Standardise → PCA → re-standardise; all fit on train only."""
    mu_r = X_tr_raw.mean(0); sd_r = X_tr_raw.std(0, ddof=0); sd_r[sd_r==0] = 1.0
    Xtr_s = (X_tr_raw - mu_r) / sd_r
    Xte_s = (X_te_raw - mu_r) / sd_r
    Xtr_pca, pca_mean, Vt = gpu_pca_fit(Xtr_s, n_components)
    Xte_pca = gpu_pca_transform(Xte_s, pca_mean, Vt)
    mu_p = Xtr_pca.mean(0); sd_p = Xtr_pca.std(0, ddof=0); sd_p[sd_p==0] = 1.0
    return (Xtr_pca - mu_p) / sd_p, (Xte_pca - mu_p) / sd_p


# ── Poisson LL ─────────────────────────────────────────────────────────────

def poisson_ll_full(Y_np, mu_np):
    """Complete Poisson LL including log-factorial, summed over words, per neuron."""
    mu = np.clip(mu_np, 1e-10, None)
    return (Y_np * np.log(mu) - mu - gammaln(Y_np + 1)).sum(0)  # (n_m,)


# ── GLM fit ────────────────────────────────────────────────────────────────

def gpu_fit(X_t, Y_t, alpha, offset_t=None, max_iter=LBFGS_ITER):
    """
    L-BFGS Poisson ridge, batched over k contexts × m neurons.
    alpha: scalar | (k,) per-batch | (m,) per-neuron (requires k=1)
    Returns beta_feat (k, m, p), bias (k, m, 1).
    """
    k, n, p = X_t.shape
    m = Y_t.shape[2]
    n_eff = float(Y_t.sum().clamp(min=1).item())
    Xi = torch.cat([torch.ones(k, n, 1, device=DEVICE), X_t], dim=-1)
    beta = torch.zeros(k, m, p + 1, device=DEVICE, requires_grad=True)
    opt  = torch.optim.LBFGS([beta], max_iter=max_iter, line_search_fn='strong_wolfe')

    if isinstance(alpha, torch.Tensor):
        if alpha.ndim == 1 and alpha.shape[0] == k:
            a = alpha.to(DEVICE).view(k, 1, 1)
        elif alpha.ndim == 1 and k == 1 and alpha.shape[0] == m:
            a = alpha.to(DEVICE).view(1, m, 1)
        else:
            a = float(alpha.item()) if alpha.numel() == 1 else float(alpha)
    else:
        a = float(alpha)

    def closure():
        opt.zero_grad()
        eta = torch.bmm(Xi, beta.transpose(1, 2))
        if offset_t is not None:
            eta = eta + offset_t
        eta  = torch.clamp(eta, -ETA_CLIP, ETA_CLIP)
        loss = ((torch.exp(eta) - Y_t * eta).sum()
                + (a * beta[:, :, 1:].pow(2)).sum()) / n_eff
        loss.backward()
        return loss

    opt.step(closure)
    b = beta.detach()
    return b[:, :, 1:], b[:, :, :1]


# ── Cross-validation splits ────────────────────────────────────────────────

def block_splits(n, k):
    b = n // k
    for i in range(k):
        s, e = i * b, (i * b + b if i < k - 1 else n)
        mask = np.zeros(n, bool); mask[s:e] = True
        yield ~mask, mask


# ── FDR ────────────────────────────────────────────────────────────────────

def fdr_bh(pvals):
    n = len(pvals)
    order = np.argsort(pvals)
    adj   = np.minimum(1.0, pvals[order] * n / np.arange(1, n + 1))
    adj   = np.minimum.accumulate(adj[::-1])[::-1]
    out   = np.empty(n); out[order] = adj
    return out, out < FDR_ALPHA


print('Helpers defined.')

## 3. Load Data

In [ ]:
# Import data loading utilities from semantic_glm.py infra
# (we re-import the functions directly for notebook clarity)
from importlib.util import spec_from_file_location, module_from_spec

_spec = spec_from_file_location(
    'spike_processing',
    os.path.join(PROJECT, 'neural_encoding', 'spike_processing.py'))
_sp = module_from_spec(_spec)
_spec.loader.exec_module(_sp)

# ── Speaker assignment ─────────────────────────────────────────────────────
xlsx_cands = [f for f in os.listdir(SPIKE_DIR) if f.endswith('_with_regress_dur.xlsx')]
assert xlsx_cands, f'No _with_regress_dur.xlsx in {SPIKE_DIR}'
tx = pd.read_excel(os.path.join(SPIKE_DIR, xlsx_cands[0]))

spk_cols = sorted(
    [c for c in tx.columns if str(c).startswith('Speaker')],
    key=lambda c: int(c.replace('Speaker','').strip())
                  if c.replace('Speaker','').strip().isdigit() else 999)

def _nn(v):
    return pd.notna(v) and str(v).strip() not in ('', 'nan')

n = len(tx)
dir_membership = {col: np.array([_nn(v) for v in tx[col]], dtype=bool) for col in spk_cols}
assign = np.array([None]*n, dtype=object)
for i in range(n):
    for col in spk_cols:
        if dir_membership[col][i]:
            assign[i] = col; break

mask_self  = assign == 'Speaker1'
mask_other = np.array([(a is not None and a != 'Speaker1') for a in assign], dtype=bool)
print(f'Words: {n} total  |  self={mask_self.sum()}  other={mask_other.sum()}')

In [ ]:
# ── Spike counts ───────────────────────────────────────────────────────────
mask = mask_self if CONDITION == 'self' else mask_other

if CONDITION == 'self':
    spk_path = os.path.join(SPIKE_DIR, 'Speaker1', f'{REGION}_spike_counts.npy')
    Y_raw = np.load(spk_path).astype(np.float32)
else:
    other_spks = sorted([c for c in dir_membership if c != 'Speaker1'])
    rows = []
    dir_pos = {s: 0 for s in other_spks}
    for i, spk in enumerate(assign):
        if spk is None or spk == 'Speaker1': continue
        for s in other_spks:
            if dir_membership[s][i]:
                p_path = os.path.join(SPIKE_DIR, s, f'{REGION}_spike_counts.npy')
                if os.path.exists(p_path):
                    m_arr = np.load(p_path)
                    if dir_pos[s] < len(m_arr):
                        rows.append(m_arr[dir_pos[s]])
                dir_pos[s] += 1; break
    Y_raw = np.vstack(rows).astype(np.float32) if rows else None

if Y_raw is None:
    raise RuntimeError('No spike data found')

# ── Filter neurons by minimum spike count ─────────────────────────────────
valid_words = ~np.isnan(Y_raw).any(axis=1)
Y_raw       = Y_raw[valid_words]
spike_ok    = Y_raw.sum(0) >= MIN_SPIKES
Y           = Y_raw[:, spike_ok]    # (n_w, n_m)
n_w, n_m    = Y.shape
print(f'Y: {n_w} words × {n_m} neurons (after spike filter)')
print(f'Mean spikes/neuron: {Y.sum(0).mean():.1f}')

In [ ]:
# ── Embeddings (raw, no global PCA) ────────────────────────────────────────
X_all = np.load(npy_path, mmap_mode='r')[LAYER].astype(np.float32)  # (n_total, D)
X_raw_c = X_all[mask][valid_words]                                    # (n_w, D)
print(f'Raw embeddings: {X_raw_c.shape}  (per-fold PCA will reduce to {N_COMPONENTS} PCs)')

# ── Poisson offset ─────────────────────────────────────────────────────────
if USE_OFFSET:
    dur_c = tx['regress_dur'].values[mask][valid_words].astype(np.float64)
    dur_c = np.where(np.isnan(dur_c) | (dur_c <= 0), 500.0, dur_c)
    offset_np = np.log(dur_c / 1000.0).astype(np.float32)   # log(seconds)
    print(f'Offset (log δ_s): min={offset_np.min():.2f}  max={offset_np.max():.2f}  '
          f'mean={offset_np.mean():.2f}  (corresponds to '
          f'{np.exp(offset_np.min())*1000:.0f}–{np.exp(offset_np.max())*1000:.0f}ms)')
else:
    offset_np = None
    print('Fixed window — no offset')

## 4. Nested CV with Per-Fold PCA

- **Outer loop** (5 folds): split into train/test blocks
- **Inner loop** (5 folds, on outer-train): select per-neuron alpha
- Per-fold PCA is fit on inner-train or outer-train only — no leakage
- Null model uses rate-based Poisson (varwin) or mean count (fixed)

In [ ]:
N_A = len(ALPHAS)
alpha_t = torch.tensor(ALPHAS, dtype=torch.float32)

ll_model_folds   = []
ll_null_folds    = []
best_alpha_folds = []
fold_meta        = []   # saved for permutation loop

t_cv = time.time()

for fold_i, (tr_m, te_m) in enumerate(block_splits(n_w, N_OUTER)):
    n_tr = int(tr_m.sum())
    print(f'Outer fold {fold_i+1}/{N_OUTER}: train={n_tr} test={int(te_m.sum())}', end=' ... ')

    # ── Per-fold PCA on outer-train ────────────────────────────────
    X_tr, X_te = prep_fold_features(X_raw_c[tr_m], X_raw_c[te_m], N_COMPONENTS)
    Y_tr = Y[tr_m]; Y_te = Y[te_m]

    off_tr = offset_np[tr_m] if offset_np is not None else None
    off_te = offset_np[te_m] if offset_np is not None else None

    # ── Null model LL ──────────────────────────────────────────────
    if USE_OFFSET and off_tr is not None:
        delta_tr = np.exp(off_tr); delta_te = np.exp(off_te)
        lambda_k = (Y_tr.sum(0) / delta_tr.sum()).clip(1e-10)
        mu_null  = lambda_k[None, :] * delta_te[:, None]
    else:
        mu_null  = np.broadcast_to(Y_tr.mean(0).clip(1e-10)[None, :], Y_te.shape).copy()
    ll_null_folds.append(poisson_ll_full(Y_te, mu_null))

    # ── Inner CV: per-neuron alpha selection ───────────────────────
    X_raw_tr = X_raw_c[tr_m]
    inner_ll = np.zeros((N_A, n_m), np.float64)

    for itr_m, iva_m in block_splits(n_tr, N_INNER):
        X_itr, X_iva = prep_fold_features(X_raw_tr[itr_m], X_raw_tr[iva_m], N_COMPONENTS)
        Y_itr = Y_tr[itr_m]; Y_iva = Y_tr[iva_m]

        Xii = torch.tensor(X_itr, device=DEVICE).unsqueeze(0).expand(N_A,-1,-1).contiguous()
        Yii = torch.tensor(Y_itr, device=DEVICE).unsqueeze(0).expand(N_A,-1,-1).contiguous()
        off_itr = None
        if off_tr is not None:
            off_itr = (torch.tensor(off_tr[itr_m][None,:,None], dtype=torch.float32, device=DEVICE)
                       .expand(N_A,-1,-1).contiguous())

        bf, bi = gpu_fit(Xii, Yii, alpha_t, offset_t=off_itr)

        with torch.no_grad():
            Xiv = torch.tensor(X_iva, device=DEVICE).unsqueeze(0).expand(N_A,-1,-1).contiguous()
            Yiv = torch.tensor(Y_iva, device=DEVICE).unsqueeze(0).expand(N_A,-1,-1).contiguous()
            off_iva = None
            if off_tr is not None:
                off_iva = (torch.tensor(off_tr[iva_m][None,:,None], dtype=torch.float32, device=DEVICE)
                           .expand(N_A,-1,-1).contiguous())
            eta_iv = torch.bmm(Xiv, bf.transpose(1,2)) + bi.transpose(1,2)
            if off_iva is not None: eta_iv = eta_iv + off_iva
            eta_iv = torch.clamp(eta_iv, -ETA_CLIP, ETA_CLIP)
            inner_ll += (Yiv * eta_iv - torch.exp(eta_iv)).sum(1).cpu().numpy()

    best_ai = np.argmax(inner_ll, axis=0)   # (n_m,)
    best_alpha_vec = torch.tensor(ALPHAS[best_ai], dtype=torch.float32)
    best_alpha_folds.append(best_ai)

    # ── Outer fit with per-neuron best alpha ───────────────────────
    X_tr_t = torch.tensor(X_tr[None], device=DEVICE)
    Y_tr_t = torch.tensor(Y_tr[None], device=DEVICE)
    off_tr_t = (torch.tensor(off_tr[None,:,None], dtype=torch.float32, device=DEVICE)
                if off_tr is not None else None)

    bf, bi = gpu_fit(X_tr_t, Y_tr_t, best_alpha_vec, offset_t=off_tr_t)

    X_te_t  = torch.tensor(X_te[None], device=DEVICE)
    off_te_t = (torch.tensor(off_te[None,:,None], dtype=torch.float32, device=DEVICE)
                if off_te is not None else None)

    with torch.no_grad():
        eta_te = torch.bmm(X_te_t, bf.transpose(1,2)) + bi.transpose(1,2)
        if off_te_t is not None: eta_te = eta_te + off_te_t
        eta_te = torch.clamp(eta_te, -ETA_CLIP, ETA_CLIP)
        mu_te  = torch.exp(eta_te[0]).cpu().numpy()

    ll_model_folds.append(poisson_ll_full(Y_te, mu_te))

    fold_meta.append({'tr_m': tr_m.copy(), 'te_m': te_m.copy(),
                      'best_alpha_vec': best_alpha_vec.clone(),
                      'off_tr': off_tr.copy() if off_tr is not None else None,
                      'off_te': off_te.copy() if off_te is not None else None,
                      'Y_te':  Y_te.copy()})

    del X_tr_t, Y_tr_t, X_te_t, bf, bi, eta_te
    torch.cuda.empty_cache()
    print(f'fold LL_model={sum(ll_model_folds[-1]):.2f}')

print(f'\nCV done in {time.time()-t_cv:.1f}s')

## 5. McFadden's Pseudo-R²

$$R^2_{McF} = 1 - \frac{\ell_{model}}{\ell_{null}}$$

Both LLs are the complete Poisson LL (including the `log(y!)` term).  
Negative R² means the model fits worse than the null.

In [ ]:
ll_model_total = sum(ll_model_folds)   # (n_m,)
ll_null_total  = sum(ll_null_folds)    # (n_m,)

with np.errstate(divide='ignore', invalid='ignore'):
    r2 = np.where(ll_null_total < -0.1,
                  1.0 - ll_model_total / ll_null_total, np.nan)

print(f'Neurons: {n_m}')
print(f'LL_model range: [{ll_model_total.min():.1f}, {ll_model_total.max():.1f}]')
print(f'LL_null  range: [{ll_null_total.min():.1f}, {ll_null_total.max():.1f}]')
print(f'R²:  median={np.nanmedian(r2):.4f}  max={np.nanmax(r2):.4f}  '
      f'frac(>0)={np.mean(r2>0):.2f}')

## 6. X-Permutation Significance Test

For each permutation:
1. Shuffle all embedding rows globally: `X_shuf = X_raw_c[rng.permutation(n_w)]`
2. Per fold: fit PCA on shuffled train, refit model with best alpha from real CV (100-iter L-BFGS)
3. Sum LL across folds to get one null LL per neuron

p-value = `(count(LL_perm >= LL_model) + 1) / (N_PERM + 1)` — matches collaborators

In [ ]:
rng = np.random.default_rng(SEED)
ll_perms = np.full((N_PERM, n_m), np.nan)

t_perm = time.time()
for pi in range(N_PERM):
    perm    = rng.permutation(n_w)
    X_shuf  = X_raw_c[perm]
    accum   = np.zeros(n_m, np.float64)

    for fd in fold_meta:
        X_str, X_ste = prep_fold_features(
            X_shuf[fd['tr_m']], X_shuf[fd['te_m']], N_COMPONENTS)
        Y_tr_f = Y[fd['tr_m']]; Y_te_f = fd['Y_te']

        X_str_t = torch.tensor(X_str[None], device=DEVICE)
        Y_tr_f_t = torch.tensor(Y_tr_f[None], device=DEVICE)
        off_tr_t = (torch.tensor(fd['off_tr'][None,:,None], dtype=torch.float32, device=DEVICE)
                    if fd['off_tr'] is not None else None)
        X_ste_t  = torch.tensor(X_ste[None], device=DEVICE)
        off_te_t = (torch.tensor(fd['off_te'][None,:,None], dtype=torch.float32, device=DEVICE)
                    if fd['off_te'] is not None else None)

        bf_p, bi_p = gpu_fit(X_str_t, Y_tr_f_t, fd['best_alpha_vec'],
                              offset_t=off_tr_t, max_iter=LBFGS_ITER_PERM)

        with torch.no_grad():
            eta_p = torch.bmm(X_ste_t, bf_p.transpose(1,2)) + bi_p.transpose(1,2)
            if off_te_t is not None: eta_p = eta_p + off_te_t
            eta_p = torch.clamp(eta_p, -ETA_CLIP, ETA_CLIP)
            mu_p  = torch.exp(eta_p[0]).cpu().numpy()

        accum += poisson_ll_full(Y_te_f, mu_p)
        del X_str_t, Y_tr_f_t, X_ste_t, bf_p, bi_p, eta_p
        torch.cuda.empty_cache()

    ll_perms[pi] = accum
    if (pi+1) % 100 == 0:
        print(f'  perm {pi+1}/{N_PERM}  {time.time()-t_perm:.0f}s', flush=True)

print(f'\nPermutation test done in {time.time()-t_perm:.1f}s')

In [ ]:
p_vals = (np.sum(ll_perms >= ll_model_total[None, :], axis=0) + 1) / (N_PERM + 1)
p_fdr, sig = fdr_bh(p_vals)

n_sig = sig.sum()
print(f'Significant (FDR {FDR_ALPHA}): {n_sig}/{n_m} ({100*n_sig/n_m:.1f}%)')
if n_sig > 0:
    print(f'  median R² (sig neurons): {np.nanmedian(r2[sig]):.4f}')
    print(f'  median LL_real: {np.median(ll_model_total[sig]):.2f}')
    print(f'  median LL_null: {np.median(ll_null_total[sig]):.2f}')

## 7. Save Results

Same format as `scripts/semantic_glm.py` output.  
Columns: `patient, region, condition, neuron_idx, layer, r2, ll_real, ll_null, p_perm, p_fdr, significant, n_spikes, best_alpha, cv_time_s`

In [ ]:
rows = []
for m in range(n_m):
    rows.append({
        'patient':    PATIENT_ID,
        'region':     REGION,
        'condition':  CONDITION,
        'neuron_idx': int(m),
        'layer':      LAYER,
        'r2':         float(r2[m]),
        'll_real':    float(ll_model_total[m]),
        'll_null':    float(ll_null_total[m]),
        'p_perm':     float(p_vals[m]),
        'p_fdr':      float(p_fdr[m]),
        'significant': bool(sig[m]),
        'n_spikes':   int(Y[:, m].sum()),
        'best_alpha': float(np.nanmedian([ALPHAS[ba[m]] for ba in best_alpha_folds])),
    })

df_results = pd.DataFrame(rows)
print(df_results.head(10).to_string())
print(f'\nTotal neurons: {len(df_results)}')

In [ ]:
import pickle

_win_suffix = '_varwin' if USE_OFFSET else ''
out_dir = os.path.join('/scratch/aniluchavez/ConvoDATAS/SemanticGLM',
                       f'{MODEL_TAG}{CONTEXT_TAG}{_win_suffix}',
                       f'pc{N_COMPONENTS}')
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, f'{PATIENT_ID}_L{LAYER:02d}_sem_notebook.pkl')
with open(out_path, 'wb') as f:
    pickle.dump({'df': df_results}, f)
print(f'Saved → {out_path}')

## 8. Quick Visualisation

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# R² distribution
ax = axes[0]
ax.hist(r2[~np.isnan(r2)], bins=40, color='steelblue', alpha=0.7)
ax.axvline(0, color='k', lw=1)
ax.set(xlabel="McFadden's R²", ylabel='Neurons', title=f'{REGION}/{CONDITION}\nR² distribution')
ax.text(0.98, 0.95, f'median={np.nanmedian(r2):.3f}', ha='right', va='top',
        transform=ax.transAxes, fontsize=9)

# p-value histogram
ax = axes[1]
ax.hist(p_vals, bins=20, color='coral', alpha=0.7)
ax.axvline(FDR_ALPHA, color='k', lw=1, ls='--', label=f'α={FDR_ALPHA}')
ax.set(xlabel='p_perm', ylabel='Neurons', title='Permutation p-values')
ax.legend(fontsize=8)

# LL model vs null (significant neurons)
ax = axes[2]
ax.scatter(ll_null_total[~sig], ll_model_total[~sig],
           s=8, alpha=0.3, color='grey', label='n.s.')
ax.scatter(ll_null_total[sig], ll_model_total[sig],
           s=15, alpha=0.7, color='steelblue', label='sig')
lims = [min(ll_null_total.min(), ll_model_total.min()),
        max(ll_null_total.max(), ll_model_total.max())]
ax.plot(lims, lims, 'k--', lw=1)
ax.set(xlabel='LL null', ylabel='LL model', title='Model vs null LL')
ax.legend(fontsize=8)

plt.suptitle(f'{PATIENT_ID}  {MODEL_TAG}{CONTEXT_TAG}  L{LAYER}  {WINDOW_TYPE}\n'
             f'sig={n_sig}/{n_m} ({100*n_sig/n_m:.1f}%)', fontsize=10)
plt.tight_layout()
plt.show()

## 9. Run Full Pipeline via Script

To run the complete pipeline for all 15 patients:

```bash
cd /scratch/aniluchavez/hippocampal-speaker-semantics
nohup python3 -u scripts/semantic_glm.py \
    --model gpt2-xl --layer 24 \
    --context_tag _ctx200 \
    --window_type varwin \
    --n_perm 500 \
    > /tmp/gpt2xl_varwin_L24_collab.log 2>&1 &
```

Monitor:
```bash
tail -f /tmp/gpt2xl_varwin_L24_collab.log
```

Results are saved to:
```
/scratch/aniluchavez/ConvoDATAS/SemanticGLM/gpt2-xl_ctx200_varwin/pc100/
  PTYEU_task147_L24_sem.pkl   # per patient
  ...
  L24_all.pkl                  # aggregated
```